# Online Boutique AIOps — 数据集读取与自动化训练/评测 Pipeline

本 notebook 演示如何使用 `online_boutique_rca_full_v1` 数据集：

1. **读取数据集**（train / valid / test 三个划分 + 隐藏的 test 答案）
2. **实现你自己的算法**（只需实现 `fit` 和 `predict` 两个方法）
3. **交给 Pipeline 自动完成**：标准化 → 训练 → 推理 → 评测 → 绘图
4. **查看结果**：所有产物按时间戳存入 `output/<timestamp>_<run_name>/`，不会互相覆盖

---

### 防泄露设计（重要）

- 数据集按**采集时间顺序**划分：train 全部早于 valid，valid 早于 test，无未来信息泄露。
- 标准化统计量（mean/std）**只在 train 上拟合**，pipeline 自动用同一组统计量变换 valid/test。
- **test 的标签不在 `processed/`**，只在 `answers/`，你的算法看不到 test 答案。

## 0. 环境准备

从项目根目录运行本 notebook（`benchmark` 包需在 import 路径上）。

In [ ]:
import sys
from pathlib import Path

# 确保能 import 到项目的 benchmark 包（notebook 在 notebooks/ 下，项目根在上一级）
ROOT = Path.cwd()
if (ROOT / "benchmark").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "benchmark").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("找不到 benchmark 包，请从项目根目录或 notebooks/ 运行")
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmark.pipeline import Pipeline, DatasetBundle, PipelineContext

DATASET = PROJECT_ROOT / "data" / "datasets" / "online_boutique_rca_full_v1"
print("项目根:", PROJECT_ROOT)
print("数据集:", DATASET)

## 1. 读取数据集

`DatasetBundle.load()` 一次性加载所有划分、答案文件、标准化统计量与元信息。

In [ ]:
bundle = DatasetBundle.load(DATASET)

print("数据集:", bundle.meta["dataset_name"])
print("划分策略:", bundle.meta["split_policy"])
print("采样间隔:", bundle.sampling_interval_seconds, "秒")
print("特征数:", len(bundle.feature_cols))
print()
print(f"train: {len(bundle.train_x):5d} 行 | 异常点 {bundle.meta['train_anomaly_points']:4d} | runs={bundle.meta['train_runs']}")
print(f"valid: {len(bundle.valid_x):5d} 行 | 异常点 {bundle.meta['valid_anomaly_points']:4d} | runs={bundle.meta['valid_runs']}")
print(f"test : {len(bundle.test_x):5d} 行 | 异常点 {bundle.meta['test_anomaly_points']:4d} | runs={bundle.meta['test_runs']}")
print(f"test incidents: {len(bundle.test_incidents)} 个故障事件（含 effect_start/end）")

In [ ]:
# 检查时序顺序：train 结束 < valid 开始 < test 开始
print("train:", bundle.train_x.timestamp.min(), "->", bundle.train_x.timestamp.max())
print("valid:", bundle.valid_x.timestamp.min(), "->", bundle.valid_x.timestamp.max())
print("test :", bundle.test_x.timestamp.min(),  "->", bundle.test_x.timestamp.max())
assert bundle.train_x.timestamp.max() < bundle.valid_x.timestamp.min() < bundle.test_x.timestamp.min()
print("\n时序顺序正确，无未来信息泄露。")

In [ ]:
# 看一眼特征与标签
display(bundle.train_x.head(3))
display(bundle.train_y.head(3))
print("标签列:", list(bundle.train_y.columns))
print("phase 取值:", bundle.train_y.phase.unique().tolist())
print("\ntest incidents 字段:", list(bundle.test_incidents.columns))
display(bundle.test_incidents[["incident_id", "fault_type", "target_service", "effect_start", "effect_end"]].head(4))

## 2. 可视化数据（可选）

看看故障期间的指标变化。cpu_stress 会抬高 recommendationservice 的 CPU 与延迟，pod_kill 会引发 cartservice 的 restart/error。

In [ ]:
ty = bundle.test_x.copy()
ty["is_anomaly"] = bundle.test_truth["y_true"].values
ts = pd.to_datetime(ty.timestamp)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(ts, ty["recommendationservice_cpu_usage"], lw=0.8)
axes[0].set_ylabel("recommend CPU")
axes[0].set_title("test 集指标（橙色 = 真实异常区间）")
axes[1].plot(ts, ty["cartservice_error_rate"], lw=0.8, color="#d62728")
axes[1].set_ylabel("cart error_rate")
for ax in axes:
    prev = 0; start = None
    for i, v in enumerate(ty.is_anomaly.values):
        if v and not prev: start = ts.iloc[i]
        elif not v and prev: ax.axvspan(start, ts.iloc[i], color="orange", alpha=0.25)
        prev = v
plt.tight_layout(); plt.show()

## 3. 实现你的算法 （⭐ 用户在这里写代码）

实现一个检测器类，只需两个方法：

```python
def fit(self, train_x, train_y, valid_x, valid_y, ctx):
    # 在这里训练。train_x/valid_x 已由 pipeline 用 train-only 统计量标准化。
    # 无监督方法可忽略 *_y。

def predict(self, test_x, ctx) -> np.ndarray:
    # 返回一维异常分数，长度 == len(test_x)，值越大越异常。
```

`ctx` 提供：`ctx.feature_cols`、`ctx.norm_stats`、`ctx.meta`、`ctx.output_dir`、`ctx.scale(df)`。

下面给出一个**示例实现**（高斯概率密度基线），你可以直接替换成自己的模型（Isolation Forest、AutoEncoder、LSTM 等）。

In [ ]:
class MyDetector:
    """示例：在标准化空间拟合对角高斯，用负对数似然作为异常分数。

    这是一个纯无监督基线：只用 train 的正常点估计均值/方差。"""

    def fit(self, train_x, train_y, valid_x, valid_y, ctx):
        self.cols = [c for c in ctx.feature_cols if c in train_x.columns]
        # 只用正常样本拟合（is_anomaly==0），更贴近异常检测设定
        normal_mask = (train_y["is_anomaly"] == 0).values
        X = train_x.loc[normal_mask, self.cols].to_numpy(dtype=float)
        self.mu = X.mean(axis=0)
        self.var = X.var(axis=0) + 1e-6

    def predict(self, test_x, ctx):
        X = test_x[self.cols].to_numpy(dtype=float)
        # 负对数高斯密度 = 0.5 * sum((x-mu)^2/var + log(var))，越大越异常
        score = 0.5 * (((X - self.mu) ** 2) / self.var + np.log(self.var)).sum(axis=1)
        return score

## 4. 运行 Pipeline

`Pipeline.run()` 自动完成：

1. 用 **train-only** 统计量标准化 train/valid/test
2. 调用你的 `fit` / `predict`
3. 选阈值（见下方 `threshold_mode`）并对照 `answers/` 计算**完整指标体系**（点级 / 排序 / 事件级 / 延迟 / 误报 / point-adjust / 分组）
4. 绘图（7 张，见第 5 节）
5. 全部产物写入 `output/<时间戳>_<run_name>/`

### 阈值模式 `threshold_mode`

| 模式 | 含义 | 可部署 |
|------|------|--------|
| `best_f1` | 在 **test** 上最大化 F1 | ❌ 仅作上界（窥探了测试标签） |
| `validation_f1` | 在 **validation** 上最大化 F1 | ✅ |
| `fixed_fpr` | 在 validation 正常点上控制目标 FPR（需 `fixed_fpr=`） | ✅ |

> `best_f1` 给出的是性能**上界**，`metrics.json` 中 `threshold_deployable=false`。无论选哪种模式，pipeline 都会额外生成 `threshold_comparison.csv/png`，并排比较 best_f1 与 validation_f1，论文汇报请以可部署结果为准。

In [ ]:
pipe = Pipeline(
    bundle,
    run_name="my_detector",
    output_root=PROJECT_ROOT / "output",
    threshold_mode="best_f1",   # 可改 "validation_f1" / "fixed_fpr"
)
result = pipe.run(MyDetector())

## 5. 查看结果

`metrics.json` 分为七大块：`point_level` / `ranking` / `event_level` / `false_alarm` / `point_adjust` / `grouped` / `threshold`。

In [ ]:
m = result.metrics

# === 重点指标速览 ===
pt, rk, ev, fa, th = m["point_level"], m["ranking"], m["event_level"], m["false_alarm"], m["threshold"]
fmt = lambda v: "null" if v is None else f"{v:.3f}"
print("================ 重点指标 ================")
print(f" Point-wise F1     : {fmt(pt['point_f1'])}")
print(f" AUPRC             : {fmt(rk['pr_auc'])}")
print(f" Event Recall      : {fmt(ev['event_recall'])}  ({ev['detected_incidents']}/{ev['detected_incidents']+ev['missed_incidents']})")
print(f" Recall@30s        : {fmt(ev['recall_at_30s'])}")
print(f" False alarms/hour : {fmt(fa['false_alarms_per_hour'])}")
print(f" Median delay (s)  : {fmt(ev['median_detection_delay_seconds'])}")
print(f" Missed incidents  : {ev['missed_incidents']}")
print(f" Threshold         : {fmt(th['threshold_value'])} [{th['threshold_mode']}, deployable={th['threshold_deployable']}]")
if m.get("warnings"):
    for w in m["warnings"]:
        print(" WARNING:", w)

In [ ]:
# === 点级 + 排序指标 ===
print("point_level:"); print(json.dumps(m["point_level"], indent=2, ensure_ascii=False))
print("\nranking:");   print(json.dumps(m["ranking"], indent=2, ensure_ascii=False))
print("\npoint_adjust:"); print(json.dumps(m["point_adjust"], indent=2, ensure_ascii=False))
print("\nfalse_alarm:"); print(json.dumps(m["false_alarm"], indent=2, ensure_ascii=False))

In [ ]:
# === 分组指标（按故障类型 / 目标服务）===
g = m["grouped"]
print("按 fault_type:")
df_ft = pd.DataFrame({
    "event_recall":  g["event_recall_by_fault_type"],
    "recall@30s":    g["recall_at_30s_by_fault_type"],
    "median_delay":  g["median_delay_by_fault_type"],
    "point_f1":      g["point_f1_by_fault_type"],
})
display(df_ft)
print("按 target_service:")
df_svc = pd.DataFrame({
    "event_recall":  g["event_recall_by_service"],
    "recall@30s":    g["recall_at_30s_by_service"],
    "median_delay":  g["median_delay_by_service"],
})
display(df_svc)

In [ ]:
# 展示生成的图表
from IPython.display import Image, display
charts = [
    "score_timeline.png",             # 分数时间线 + 真实异常带
    "roc_pr_curves.png",              # ROC / PR 曲线
    "score_distribution.png",         # 正常 vs 异常分数分布
    "incident_delay_bar.png",         # 5.1 逐 incident 检测延迟（漏检红叉）
    "event_recall_by_fault_type.png", # 5.2 按故障类型的 Event Recall / Recall@30s
    "false_positive_timeline.png",    # 5.3 误报时间线
    "threshold_comparison.png",       # 5.4 best_f1 vs validation_f1 对比
]
for name in charts:
    p = result.output_dir / name
    if p.exists():
        print(name)
        display(Image(str(p)))

In [ ]:
# 阈值模式对比表（best_f1 上界 vs validation_f1 可部署）
cmp_path = result.output_dir / "threshold_comparison.csv"
if cmp_path.exists():
    display(pd.read_csv(cmp_path))

In [ ]:
# 逐 incident 检出 + 延迟明细
per_inc = pd.read_csv(result.output_dir / "per_incident.csv")
print(f"检出 {per_inc.detected.sum()}/{len(per_inc)} 个 incident")
display(per_inc.head(12))

## 6. 反复实验

每次 `pipe.run()` 都会创建一个新的时间戳目录，不会覆盖历史结果。你可以换不同算法/参数/阈值模式多次运行，然后对比 `output/` 下的各次结果。

### 离线提交方式

如果你在别处跑出了预测分数，也可以直接提交一个 `submission` DataFrame（参考 `examples/sample_submission.csv`），跳过 fit/predict：

```python
sub = pd.read_csv(DATASET / "examples" / "sample_submission.csv")
sub["anomaly_score"] = my_scores   # 填入你的分数
result = pipe.run(detector=None, submission=sub)
```

In [ ]:
# 列出 output/ 下所有历史运行
out_root = PROJECT_ROOT / "output"
rows = []
if out_root.exists():
    for d in sorted(out_root.iterdir()):
        mj = d / "metrics.json"
        if d.is_dir() and mj.exists():
            mm = json.loads(mj.read_text(encoding="utf-8"))
            rows.append({
                "run": d.name,
                "point_f1": mm["point_level"]["point_f1"],
                "pr_auc": mm["ranking"]["pr_auc"],
                "event_recall": mm["event_level"]["event_recall"],
                "recall@30s": mm["event_level"]["recall_at_30s"],
                "far/h": mm["false_alarm"]["false_alarms_per_hour"],
                "median_delay": mm["event_level"]["median_detection_delay_seconds"],
                "thr_mode": mm["threshold"]["threshold_mode"],
                "deployable": mm["threshold"]["threshold_deployable"],
            })
pd.DataFrame(rows)